1/ Import data from github shared repository

In [33]:
#Import packages

import pandas as pd
import statsmodels.api as sm

In [35]:
#Load the dataset from GitHub

#USe CPI dates from 1958 to Febraury 1972
CPI_announcement_dates = "https://raw.githubusercontent.com/carolinebazeli/Memoire_HEC/refs/heads/master/CPI_announcement_date.csv"
CPI_announcement_dates = pd.read_csv(CPI_announcement_dates)

#Use PPI dates from 1972 to 2024
PPI_announcement_dates = "https://raw.githubusercontent.com/carolinebazeli/Memoire_HEC/refs/heads/master/PPI_announcement_date.csv"
PPI_announcement_dates = pd.read_csv(PPI_announcement_dates)

#FOMC scheduled interest rate announcement dates from 1990 to 2024
FOMC_announcement_dates = "https://raw.githubusercontent.com/carolinebazeli/Memoire_HEC/refs/heads/master/FOMC_announcement_date.csv"
FOMC_announcement_dates = pd.read_csv(FOMC_announcement_dates, nrows=106)

#Stock market proxy - CRPS NYSE value-weighted index of all listed shares
NYSE_daily_stock_returns = r"C:/Users/bazel/Documents/Master in Finance/Mémoire/Data/Stocks/NYSE_daily_stock_returns.csv"
NYSE_daily_stock_returns = pd.read_csv(NYSE_daily_stock_returns)

#Stock market proxy - CRPS Amex value-weighted index of all listed shares
Amex_daily_stock_returns = r"C:/Users/bazel/Documents/Master in Finance/Mémoire/Data/Stocks/Amex_daily_stock_returns.csv"
Amex_daily_stock_returns = pd.read_csv(Amex_daily_stock_returns)

#Fama_French_25_portfolio_daily (Equal_Weighted=EV, Value_Weighted=VW)
Fama_French_25_portfolio_daily ="https://raw.githubusercontent.com/carolinebazeli/Memoire_HEC/refs/heads/master/25_Portfolios_Size_BM_Daily.csv"
Fama_French_25_portfolio_daily_EV = pd.read_csv(Fama_French_25_portfolio_daily, skiprows=18, nrows=25920-19)
Fama_French_25_portfolio_daily_VW = pd.read_csv(Fama_French_25_portfolio_daily, skiprows=25923, nrows=51825-25924)

#Ten_industry_portfolio_daily (Equal_Weighted=EV, Value_Weighted=VW)
Ten_industry_portfolio_daily = "https://raw.githubusercontent.com/carolinebazeli/Memoire_HEC/refs/heads/master/10_Industry_Portfolios_Daily.csv"
Ten_industry_portfolio_daily_EV = pd.read_csv(Ten_industry_portfolio_daily, skiprows=9, nrows=25911 - 10)
Ten_industry_portfolio_daily_VW = pd.read_csv(Ten_industry_portfolio_daily, skiprows=25914, nrows=51816 - 25915)

#Fama_French_3_factor_daily
Fama_French_3_factor_daily = "https://raw.githubusercontent.com/carolinebazeli/Memoire_HEC/refs/heads/master/F-F_3_Factors_Daily.CSV"
Fama_French_3_factor_daily = pd.read_csv(Fama_French_3_factor_daily, skiprows=4, nrows=25906 - 5)

C:\Users\bazel\AppData\Local\Temp\ipykernel_24192\3111651582.py:17: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  NYSE_daily_stock_returns = pd.read_csv(NYSE_daily_stock_returns)


In [39]:
#Convert NYSE, AMex stock returns to a Dataframe, where each column is a stock and each row is a date, and the value is the daily return of that stock

NYSE_daily_stock_returns = NYSE_daily_stock_returns.drop(columns=['PERMNO','EXCHCD'])
NYSE_daily_stock_returns['RET'] = pd.to_numeric(NYSE_daily_stock_returns['RET'], errors='coerce')
NYSE_daily_stock_returns = NYSE_daily_stock_returns.groupby(['date', 'TICKER'])['RET'].mean().reset_index()
NYSE_daily_stock_returns = NYSE_daily_stock_returns.pivot(index='date', columns='TICKER', values='RET').reset_index()

Amex_daily_stock_returns = Amex_daily_stock_returns.drop(columns=['PERMNO','EXCHCD'])
Amex_daily_stock_returns['RET'] = pd.to_numeric(Amex_daily_stock_returns['RET'], errors='coerce')
Amex_daily_stock_returns = Amex_daily_stock_returns.groupby(['date', 'TICKER'])['RET'].mean().reset_index()
Amex_daily_stock_returns = Amex_daily_stock_returns.pivot(index='date', columns='TICKER', values='RET').reset_index()


In [45]:
#Convert date columns to datetime format
CPI_announcement_dates = CPI_announcement_dates.rename(columns = {'Release Dates': 'A_date'})
CPI_announcement_dates['A_date'] = pd.to_datetime(CPI_announcement_dates['A_date']).dt.date

PPI_announcement_dates = PPI_announcement_dates.rename(columns = {'Release Dates': 'A_date'})
PPI_announcement_dates['A_date'] = pd.to_datetime(PPI_announcement_dates['A_date']).dt.date

FOMC_announcement_dates = FOMC_announcement_dates.rename(columns = {'FOMC Meeting Date': 'A_date'}).reset_index(drop=True)
FOMC_announcement_dates['A_date'] = pd.to_datetime(FOMC_announcement_dates['A_date']).dt.date

NYSE_daily_stock_returns = NYSE_daily_stock_returns.rename(columns = {'date': 'Date'})
NYSE_daily_stock_returns['Date'] = pd.to_datetime(NYSE_daily_stock_returns['Date'], errors='coerce').dt.date

Amex_daily_stock_returns = Amex_daily_stock_returns.rename(columns = {'date': 'Date'})
Amex_daily_stock_returns['Date'] = pd.to_datetime(Amex_daily_stock_returns['Date'], errors='coerce').dt.date

Fama_French_25_portfolio_daily_EV = Fama_French_25_portfolio_daily_EV.rename(columns = {'Unnamed: 0': 'Date'})
Fama_French_25_portfolio_daily_EV['Date'] = pd.to_datetime(Fama_French_25_portfolio_daily_EV['Date'].astype(str), format='%Y%m%d').dt.date

Fama_French_25_portfolio_daily_VW = Fama_French_25_portfolio_daily_VW.rename(columns = {'Unnamed: 0': 'Date'})
Fama_French_25_portfolio_daily_VW['Date'] = pd.to_datetime(Fama_French_25_portfolio_daily_VW['Date'].astype(str), format='%Y%m%d').dt.date

Ten_industry_portfolio_daily_EV = Ten_industry_portfolio_daily_EV.rename(columns = {'Unnamed: 0': 'Date'})
Ten_industry_portfolio_daily_EV['Date'] = pd.to_datetime(Ten_industry_portfolio_daily_EV['Date'].astype(str), format='%Y%m%d').dt.date

Ten_industry_portfolio_daily_VW = Ten_industry_portfolio_daily_VW.rename(columns = {'Unnamed: 0': 'Date'})
Ten_industry_portfolio_daily_VW['Date'] = pd.to_datetime(Ten_industry_portfolio_daily_VW['Date'].astype(str), format='%Y%m%d').dt.date

Fama_French_3_factor_daily = Fama_French_3_factor_daily.rename(columns = {'Unnamed: 0': 'Date'})
Fama_French_3_factor_daily['Date'] = pd.to_datetime(Fama_French_3_factor_daily['Date'].astype(str), format='%Y%m%d').dt.date

C:\Users\bazel\AppData\Local\Temp\ipykernel_24192\2478066437.py:9: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  FOMC_announcement_dates['A_date'] = pd.to_datetime(FOMC_announcement_dates['A_date']).dt.date


2/ Split data between announcement days and non-announcement days for each portfolio

In [46]:
#Select CPI_announcement_dates up until February 1972
CPI_announcement_dates = CPI_announcement_dates.loc[CPI_announcement_dates['A_date'] <= pd.to_datetime('1972-02-01').date()]
CPI_announcement_dates = CPI_announcement_dates.reset_index(drop=True)

#Select PPI_announcement_dates from February 1972
PPI_announcement_dates = PPI_announcement_dates.loc[PPI_announcement_dates['A_date'] > pd.to_datetime('1972-02-01').date()]
PPI_announcement_dates = PPI_announcement_dates.reset_index(drop=True)

In [47]:
all_A_dates = pd.concat([
    CPI_announcement_dates['A_date'],
    FOMC_announcement_dates['A_date'],
    PPI_announcement_dates['A_date']
])

NYSE_daily_stock_returns_A_Day = NYSE_daily_stock_returns[NYSE_daily_stock_returns['Date'].isin(all_A_dates)].reset_index(drop=True)
NYSE_daily_stock_returns_N_Day = NYSE_daily_stock_returns[~NYSE_daily_stock_returns['Date'].isin(all_A_dates)].reset_index(drop=True)

Fama_French_25_portfolio_daily_EV_A_Day = Fama_French_25_portfolio_daily_EV[Fama_French_25_portfolio_daily_EV['Date'].isin(all_A_dates)].reset_index(drop=True)
Fama_French_25_portfolio_daily_VW_A_Day = Fama_French_25_portfolio_daily_VW[Fama_French_25_portfolio_daily_VW['Date'].isin(all_A_dates)].reset_index(drop=True)
Ten_industry_portfolio_daily_VW_A_Day = Ten_industry_portfolio_daily_VW[Ten_industry_portfolio_daily_VW['Date'].isin(all_A_dates)].reset_index(drop=True)
Ten_industry_portfolio_daily_EV_A_Day = Ten_industry_portfolio_daily_EV[Ten_industry_portfolio_daily_EV['Date'].isin(all_A_dates)].reset_index(drop=True)

Fama_French_25_portfolio_daily_EV_N_Day = Fama_French_25_portfolio_daily_EV[~Fama_French_25_portfolio_daily_EV['Date'].isin(all_A_dates)].reset_index(drop=True)
Fama_French_25_portfolio_daily_VW_N_Day = Fama_French_25_portfolio_daily_VW[~Fama_French_25_portfolio_daily_VW['Date'].isin(all_A_dates)].reset_index(drop=True)
Ten_industry_portfolio_daily_VW_N_Day = Ten_industry_portfolio_daily_VW[~Ten_industry_portfolio_daily_VW['Date'].isin(all_A_dates)].reset_index(drop=True)
Ten_industry_portfolio_daily_EV_N_Day = Ten_industry_portfolio_daily_EV[~Ten_industry_portfolio_daily_EV['Date'].isin(all_A_dates)].reset_index(drop=True)


In [48]:
#add the daily market Equity Risk premium and risk-free rate to all the portfolios

Fama_French_3_factor_daily = Fama_French_3_factor_daily.rename(columns={'Mkt-RF': 'Equity Risk Premium', 'RF': 'Risk-Free Rate'})

df_names = [
    'NYSE_daily_stock_returns',
    'NYSE_daily_stock_returns_A_Day',
    'NYSE_daily_stock_returns_N_Day',
    'Fama_French_25_portfolio_daily_EV',
    'Fama_French_25_portfolio_daily_VW',
    'Ten_industry_portfolio_daily_VW',
    'Ten_industry_portfolio_daily_EV',
    'Fama_French_25_portfolio_daily_EV_A_Day',
    'Fama_French_25_portfolio_daily_VW_A_Day',
    'Ten_industry_portfolio_daily_VW_A_Day',
    'Ten_industry_portfolio_daily_EV_A_Day',
    'Fama_French_25_portfolio_daily_EV_N_Day',
    'Fama_French_25_portfolio_daily_VW_N_Day',
    'Ten_industry_portfolio_daily_VW_N_Day',
    'Ten_industry_portfolio_daily_EV_N_Day'
]

ff3_cols = ['Date', 'Equity Risk Premium', 'Risk-Free Rate']

for name in df_names:
    globals()[name] = globals()[name].merge(Fama_French_3_factor_daily[ff3_cols], on='Date', how='left')

Mathod 1: Classic Two-Step CAPM with Full-Sample Beta

In [49]:
#Step 1a - Estimate unconditionnal full-sample beta through a time-series regression - run CAPM regression on all daily data for each stock

import statsmodels.api as sm

def compute_rolling_betas_from_dataset(data, window=252):
    """
    Compute 1-year rolling betas using a dataset where:
    - rows are dates,
    - columns: [stock1, ..., stockN, 'Equity Risk Premium', 'Risk-Free Rate']

    Returns:
    - DataFrame with MultiIndex (date, ticker) and column ['beta']
    """
    required_columns = ['Equity Risk Premium', 'Risk-Free Rate']
    stock_columns = data.columns.difference(required_columns)
    
    betas = []
    end_of_months = data.resample('M').last().index

    for current_date in end_of_months:
        try:
            start_idx = data.index.get_loc(current_date)
        except KeyError:
            continue

        if start_idx < window:
            continue
        
        window_data = data.iloc[start_idx - window:start_idx]

        if len(window_data) < 200:
            continue

        X = sm.add_constant(window_data['Equity Risk Premium'])

        for stock in stock_columns:
            y = window_data[stock] - window_data['Risk-Free Rate']

            if y.isnull().sum() > 50:
                continue

            try:
                model = sm.OLS(y, X, missing='drop')
                results = model.fit()
                beta = results.params['Equity Risk Premium']
                betas.append((current_date, stock, beta))
            except:
                continue

    betas_time_varying = pd.DataFrame(betas, columns=['date', 'ticker', 'beta'])
    return betas_time_varying.pivot(index='date', columns='ticker', values='beta').sort_index()


In [ ]:
#Step 1b - Estimate One-Year rolling window Betas (Time-Varying betas) on all daily data for each stock (unconditionnal betas)

import numpy as np
from tqdm import tqdm

def rolling_betas(data, window=252):
    """
    Compute 1-year rolling betas using a dataset where:
    - rows are dates (index),
    - columns: [stock1, ..., stockN, 'Equity Risk Premium', 'Risk-Free Rate']

    Returns:
    - DataFrame with MultiIndex (date, ticker) and column ['beta']
    """
    data.set_index('Date', inplace=True)
    data.index = pd.to_datetime(data.index, errors='coerce')
    
    required_columns = ['Equity Risk Premium', 'Risk-Free Rate']
    stock_columns = data.columns.difference(required_columns)
    
    betas = []
    
    # Resample to get the end of each month
            #end_of_months = data.resample('M').last().index
            #end_of_months = pd.date_range(start=data.index.min(), end=data.index.max(), freq='BM')
            #end_of_months = [d for d in end_of_months if d in data.index]

    end_of_months = data.groupby([data.index.year, data.index.month]).apply(lambda x: x.index.max()).sort_values()
    
    for current_date in tqdm(end_of_months, desc='Rolling beta estimation'):
        
        try:
            start_idx = data.index.get_loc(current_date)
        except KeyError:
            continue

        if start_idx < window:
            continue
        
        window_data = data.iloc[start_idx - window:start_idx]

        if len(window_data) < 200:
            continue

        X = sm.add_constant(window_data['Equity Risk Premium'])

        for stock in stock_columns:
            y = window_data[stock] - window_data['Risk-Free Rate']

            if y.isnull().sum() > 50:
                continue

            try:
                model = sm.OLS(y, X, missing='drop')
                results = model.fit()
                beta = results.params['Equity Risk Premium']
                betas.append((current_date, stock, beta))
            except:
                continue
              
    betas_time_varying = pd.DataFrame(betas, columns=['date', 'ticker', 'beta'])
    return betas_time_varying.set_index(['date', 'ticker'])['beta'].unstack()


3/ Estimate unconditonal full sample beta

In [ ]:
#We estimate the full sample market beta, on all dates (unconditonnaly to a-date and n-date), for each portfolio
df_names = [
    'Fama_French_25_portfolio_daily_EV',
    'Fama_French_25_portfolio_daily_VW',
    'Ten_industry_portfolio_daily_VW',
    'Ten_industry_portfolio_daily_EV'
]

regression_results = {}

for name in df_names:

    df = globals()[name]
    if name not in regression_results:
            regression_results[name] = {}

    print(f"\n=== Running regressions on: {name} ===")
    
    for col in df.columns:
        if col in ['Date', 'Mkt-RF', 'RF']:
            continue

        print(f"\n Regression for: {col}\n")

        # Dependent variable: portfolio excess return
        y = df[col] - df['RF']
        
        # Independent variable: market excess return
        X = sm.add_constant(df['Mkt-RF'])

        # Run OLS regression
        model = sm.OLS(y, X).fit()

        alpha = model.params['const']
        beta = model.params['Mkt-RF']

        regression_results[name][col] = {
            'alpha': model.params['const'],
            'beta': model.params['Mkt-RF']
        }


In [ ]:
print(regression_results)

In [ ]:
#Compute daily average excess returns for each portfolio

average_excess_returns = {
    'a_days': {},
    'n_days': {}
}

# List of portfolios to process
portfolio_df_names = {
    'a_days': Fama_French_25_portfolio_daily_EV_A_Day,
    'n_days': Fama_French_25_portfolio_daily_EV_N_Day
}

for period, df in portfolio_df_names.items():
    for col in df.columns:
        if col in ['Date', 'Mkt-RF', 'RF']:
            continue

        excess_returns = df[col] - df['RF']
        avg_excess = excess_returns.mean()

        average_excess_returns[period][col] = avg_excess